Installations

In [ ]:
!python -m venv agrigemma_env







In [ ]:
!agrigemma_env\Scripts\activate

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

In [ ]:
!pip install transformers accelerate sentencepiece

In [ ]:
!python -m pip install --upgrade pip setuptools wheel

In [7]:
!pip install pandas matplotlib seaborn scikit-learn flask flask_sqlalchemy flask_marshmallow marshmallow-sqlalchemy flask-cors gunicorn

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 2.0 MB/s eta 0:00:05
   ---- ----------------------------------- 1.0/9.9 MB 1.7 MB/s eta 0:00:06
   ----- ---------------------------------- 1.3/9.9 MB 1.7 MB/s eta 0:00:06
   ------ --------------------------------- 1.6/9.9 MB 1.7 MB/s eta 0:00:05
   -------- ------------------------------- 2.1/9.9 MB 1.6 MB/s eta 0:00:05
   --------- ------------------------------ 2.4/9.9 MB 1.6 MB/s eta 0:00:05
   ---------- ----------------------------- 2.6/9.9 MB 1.6 MB/s eta 0:00:05
   ----------- ---------------------------- 2.9/9.9 MB 1.6 MB/s eta 0:00:05
   ------------- -------------------------- 3.4/9.9 MB 1.6 MB/s eta 0:00:04
   -------------- ------------------------- 3.7/9.9 MB 1.6 MB/s eta 0:00:04
   --------------- ------------------------ 3.9/9.9 MB 1.6 MB/s eta 0:00:04
   ----------------- -----

In [ ]:
!pip -q install  -U transformers accelerate bitsandbytes sentencepiece

In [8]:
import json
import pandas as pd
from datetime import datetime, timedelta

Sample Farm Input

In [9]:
farm_profile = {
    "farmer_name": "Demo Farmer",
    "location": "Punjab, Pakistan",
    "crop": "maize",
    "soil_type": "loam",
    "irrigation_available": True,
    "field_size_acres": 5,
    "planned_sowing_date": "2026-06-10",
    "current_growth_stage": "pre-sowing"
}

forecast_data = {
    "next_7_days": [
        {"date": "2026-06-09", "rain_mm": 22, "max_temp_c": 35, "humidity": 70},
        {"date": "2026-06-10", "rain_mm": 28, "max_temp_c": 34, "humidity": 76},
        {"date": "2026-06-11", "rain_mm": 18, "max_temp_c": 33, "humidity": 74},
        {"date": "2026-06-12", "rain_mm": 8,  "max_temp_c": 36, "humidity": 65},
        {"date": "2026-06-13", "rain_mm": 1,  "max_temp_c": 39, "humidity": 49},
        {"date": "2026-06-14", "rain_mm": 0,  "max_temp_c": 40, "humidity": 42},
        {"date": "2026-06-15", "rain_mm": 0,  "max_temp_c": 41, "humidity": 39},
    ]
}

Crop Rule Base

In [10]:
crop_rules = {
    "maize": {
        "ideal_sowing_temp_min": 20,
        "ideal_sowing_temp_max": 35,
        "heavy_rain_threshold_mm": 15,
        "heat_stress_temp_c": 38,
        "fertilizer_avoid_rain_mm": 10
    },
    "wheat": {
        "ideal_sowing_temp_min": 15,
        "ideal_sowing_temp_max": 25,
        "heavy_rain_threshold_mm": 20,
        "heat_stress_temp_c": 32,
        "fertilizer_avoid_rain_mm": 8
    }
}

Risk Calculator

In [11]:
def compute_weather_risk(crop, forecast, rules):
    r = rules[crop]
    heavy_rain_days = []
    heatwave_days = []

    for day in forecast["next_7_days"]:
        if day["rain_mm"] >= r["heavy_rain_threshold_mm"]:
            heavy_rain_days.append(day["date"])
        if day["max_temp_c"] >= r["heat_stress_temp_c"]:
            heatwave_days.append(day["date"])

    risk_summary = {
        "heavy_rain_risk": len(heavy_rain_days) > 0,
        "heavy_rain_days": heavy_rain_days,
        "heat_stress_risk": len(heatwave_days) > 0,
        "heatwave_days": heatwave_days
    }
    return risk_summary

Initial planner

In [12]:
def generate_initial_plan(profile):
    sowing_date = datetime.strptime(profile["planned_sowing_date"], "%Y-%m-%d")
    plan = {
        "sowing": sowing_date.strftime("%Y-%m-%d"),
        "first_irrigation": (sowing_date + timedelta(days=8)).strftime("%Y-%m-%d"),
        "fertilizer_application": (sowing_date + timedelta(days=15)).strftime("%Y-%m-%d"),
        "harvest_window_start": (sowing_date + timedelta(days=110)).strftime("%Y-%m-%d"),
        "harvest_window_end": (sowing_date + timedelta(days=120)).strftime("%Y-%m-%d")
    }
    return plan

Plan Updater

In [13]:
def update_plan_from_forecast(profile, plan, risk_summary):
    updated_plan = plan.copy()
    actions = []

    sowing_date = datetime.strptime(plan["sowing"], "%Y-%m-%d")

    if risk_summary["heavy_rain_risk"]:
        updated_sowing = sowing_date + timedelta(days=4)
        updated_plan["sowing"] = updated_sowing.strftime("%Y-%m-%d")
        updated_plan["first_irrigation"] = (updated_sowing + timedelta(days=8)).strftime("%Y-%m-%d")
        updated_plan["fertilizer_application"] = (updated_sowing + timedelta(days=15)).strftime("%Y-%m-%d")
        actions.append("Delay sowing by 4 days due to heavy rainfall risk.")
        actions.append("Inspect field drainage before sowing.")

    if risk_summary["heat_stress_risk"]:
        actions.append("Prepare earlier irrigation due to forecast heat stress.")
        actions.append("Monitor crop moisture closely during hot days.")

    return updated_plan, actions

Running the planner

In [14]:
risk_summary = compute_weather_risk(
    crop=farm_profile["crop"],
    forecast=forecast_data,
    rules=crop_rules
)

initial_plan = generate_initial_plan(farm_profile)
updated_plan, actions = update_plan_from_forecast(
    farm_profile,
    initial_plan,
    risk_summary
)

print("Risk Summary:")
print(json.dumps(risk_summary, indent=2))

print("\nInitial Plan:")
print(json.dumps(initial_plan, indent=2))

print("\nUpdated Plan:")
print(json.dumps(updated_plan, indent=2))

print("\nRecommended Actions:")
for a in actions:
    print("-", a)

Risk Summary:
{
  "heavy_rain_risk": true,
  "heavy_rain_days": [
    "2026-06-09",
    "2026-06-10",
    "2026-06-11"
  ],
  "heat_stress_risk": true,
  "heatwave_days": [
    "2026-06-13",
    "2026-06-14",
    "2026-06-15"
  ]
}

Initial Plan:
{
  "sowing": "2026-06-10",
  "first_irrigation": "2026-06-18",
  "fertilizer_application": "2026-06-25",
  "harvest_window_start": "2026-09-28",
  "harvest_window_end": "2026-10-08"
}

Updated Plan:
{
  "sowing": "2026-06-14",
  "first_irrigation": "2026-06-22",
  "fertilizer_application": "2026-06-29",
  "harvest_window_start": "2026-09-28",
  "harvest_window_end": "2026-10-08"
}

Recommended Actions:
- Delay sowing by 4 days due to heavy rainfall risk.
- Inspect field drainage before sowing.
- Prepare earlier irrigation due to forecast heat stress.
- Monitor crop moisture closely during hot days.


Gemma4 prompting layer


In [15]:
planning_context = {
    "farm_profile": farm_profile,
    "risk_summary": risk_summary,
    "initial_plan": initial_plan,
    "updated_plan": updated_plan,
    "recommended_actions": actions
}

prompt = f"""
You are an agricultural planning assistant helping a small farmer.

Given the following farm data, compare the initial seasonal plan with the updated plan.
Explain:
1. What changed
2. Why it changed
3. What the farmer should do this week
4. The urgency level
5. A short farmer-friendly advisory in simple English

Return the answer in JSON with keys:
changed_plan, reason, weekly_actions, urgency, farmer_advisory

DATA:
{json.dumps(planning_context, indent=2)}
"""

Inference Skeleton

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "google/gemma-4-e4b-it"   # adjust if needed

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=500,
    temperature=0.3
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

c:\Users\its\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\its\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

Judge Frindly Output

In [ ]:
prompt = f"""
You are an agricultural planning AI.

Analyze the farm situation and return valid JSON only.

Schema:
{{
  "changed_plan": ["string"],
  "reason": "string",
  "weekly_actions": ["string"],
  "urgency": "low|medium|high",
  "farmer_advisory": "string"
}}

Farm data:
{json.dumps(planning_context, indent=2)}
"""

Demo Scenarios

In [4]:
scenario_forecasts = {
    "Scenario 1 - Heavy rain before sowing": {
        "next_7_days": [
            {"date": "2026-06-09", "rain_mm": 22, "max_temp_c": 35, "humidity": 70},
            {"date": "2026-06-10", "rain_mm": 28, "max_temp_c": 34, "humidity": 76},
            {"date": "2026-06-11", "rain_mm": 18, "max_temp_c": 33, "humidity": 74},
            {"date": "2026-06-12", "rain_mm": 8,  "max_temp_c": 36, "humidity": 65},
            {"date": "2026-06-13", "rain_mm": 1,  "max_temp_c": 39, "humidity": 49},
            {"date": "2026-06-14", "rain_mm": 0,  "max_temp_c": 40, "humidity": 42},
            {"date": "2026-06-15", "rain_mm": 0,  "max_temp_c": 41, "humidity": 39},
        ]
    },

    "Scenario 2 - Heatwave after sowing": {
        "next_7_days": [
            {"date": "2026-06-09", "rain_mm": 0, "max_temp_c": 36, "humidity": 35},
            {"date": "2026-06-10", "rain_mm": 0, "max_temp_c": 38, "humidity": 30},
            {"date": "2026-06-11", "rain_mm": 0, "max_temp_c": 40, "humidity": 28},
            {"date": "2026-06-12", "rain_mm": 0, "max_temp_c": 41, "humidity": 25},
            {"date": "2026-06-13", "rain_mm": 0, "max_temp_c": 42, "humidity": 24},
            {"date": "2026-06-14", "rain_mm": 0, "max_temp_c": 39, "humidity": 29},
            {"date": "2026-06-15", "rain_mm": 0, "max_temp_c": 37, "humidity": 32},
        ]
    },

    "Scenario 3 - Rain near harvest": {
        "next_7_days": [
            {"date": "2026-09-25", "rain_mm": 0,  "max_temp_c": 31, "humidity": 45},
            {"date": "2026-09-26", "rain_mm": 5,  "max_temp_c": 30, "humidity": 50},
            {"date": "2026-09-27", "rain_mm": 18, "max_temp_c": 29, "humidity": 70},
            {"date": "2026-09-28", "rain_mm": 24, "max_temp_c": 28, "humidity": 78},
            {"date": "2026-09-29", "rain_mm": 20, "max_temp_c": 27, "humidity": 80},
            {"date": "2026-09-30", "rain_mm": 10, "max_temp_c": 29, "humidity": 72},
            {"date": "2026-10-01", "rain_mm": 2,  "max_temp_c": 30, "humidity": 60},
        ]
    }
}

In [5]:
def run_scenario(scenario_name, farm_profile, forecast_data, crop_rules):
    print("\n" + "="*60)
    print(scenario_name)
    print("="*60)

    risk_summary = compute_weather_risk(
        crop=farm_profile["crop"],
        forecast=forecast_data,
        rules=crop_rules
    )

    initial_plan = generate_initial_plan(farm_profile)
    updated_plan, actions = update_plan_from_forecast(
        farm_profile,
        initial_plan,
        risk_summary
    )

    print("\nRisk Summary:")
    print(json.dumps(risk_summary, indent=2))

    print("\nInitial Plan:")
    print(json.dumps(initial_plan, indent=2))

    print("\nUpdated Plan:")
    print(json.dumps(updated_plan, indent=2))

    print("\nRecommended Actions:")
    for action in actions:
        print("-", action)

In [6]:
for scenario_name, forecast in scenario_forecasts.items():
    run_scenario(scenario_name, farm_profile, forecast, crop_rules)

NameError: name 'farm_profile' is not defined